# Proyek Pengembangan Machine Learning Pipeline (Dicoding Submission 1)

- **Nama**: Sonny Ariady
- **Username Dicoding**: sonnyariady
- **Dataset**: Heart Disease Dataset (Binary Classification - Health)
- **Komponen TFX**: ExampleGen, StatisticGen, SchemaGen, ExampleValidator, Transform, Tuner, Trainer, Resolver, Evaluator, Pusher

Notebook ini mengimplementasikan Machine Learning Pipeline terautomasi secara end-to-end berbasis **TensorFlow Extended (TFX)** menggunakan `InteractiveContext`.

## 1. Import Library & Pengaturan Environment

In [1]:
import os
import sys
import tensorflow as tf
from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Tuner,
    Trainer,
    Evaluator,
    Pusher
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
import tensorflow_model_analysis as tfma

print(f"TensorFlow Version: {tf.__version__}")


C:\Latihan\AI\Dicoding\MachineLearningPipeline\venv\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


TensorFlow Version: 2.13.1


## 2. Inisialisasi InteractiveContext & Directory Pipeline

In [2]:
USERNAME = "sonnyariady"
PIPELINE_NAME = f"{USERNAME}-pipeline"
PIPELINE_ROOT = PIPELINE_NAME
DATA_ROOT = "data"
SERVING_MODEL_DIR = os.path.join("serving_model", "heart-disease-model")

# Inisialisasi InteractiveContext (menggunakan path C:/t pada Windows untuk menghindari MAX_PATH limit TFX)
if os.name == 'nt':
    temp_root = "C:/t"
    os.makedirs(temp_root, exist_ok=True)
    context = InteractiveContext(pipeline_root=temp_root)
else:
    context = InteractiveContext(pipeline_root=PIPELINE_ROOT)

print(f"InteractiveContext initialized at: {PIPELINE_ROOT}")


InteractiveContext initialized at: sonnyariady-pipeline


## 3. Komponen 1: ExampleGen
`ExampleGen` membaca dataset CSV dari direktori `data/` dan membaginya menjadi data latih (train) dan evaluasi (eval) dalam format TFRecord.

In [3]:
example_gen = CsvExampleGen(input_base=DATA_ROOT)
context.run(example_gen)


ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 21
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 4. Komponen 2: StatisticGen
`StatisticGen` mengomputasi statistik deskriptif untuk data latih dan evaluasi.

In [4]:
statistics_gen = StatisticsGen(
    examples=example_gen.outputs['examples']
)
context.run(statistics_gen)
context.show(statistics_gen.outputs['statistics'])


## 5. Komponen 3: SchemaGen
`SchemaGen` menginferensi skema data berdasarkan hasil statistik (tipe data, rentang nilai, ketersediaan fitur).

In [5]:
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs['statistics'],
    infer_feature_shape=True
)
context.run(schema_gen)
context.show(schema_gen.outputs['schema'])


,Type,Presence,Valency,Domain
Feature name,,,,
'age',INT,required,,-
'ca',INT,required,,-
'chol',INT,required,,-
'cp',INT,required,,-
'exang',INT,required,,-
'fbs',INT,required,,-
'oldpeak',FLOAT,required,,-
'restecg',INT,required,,-
'sex',INT,required,,-


## 6. Komponen 4: ExampleValidator
`ExampleValidator` mendeteksi adanya anomali atau penyimpangan data terhadap skema yang telah dibuat.

In [6]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
context.run(example_validator)
context.show(example_validator.outputs['anomalies'])


## 7. Komponen 5: Transform
`Transform` melakukan preprocessing fitur data (normalisasi fitur numerik dan pencocokan vocabulary fitur kategorikal) menggunakan modul `transform.py`.

In [7]:
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file='transform.py'
)
context.run(transform)


INFO:tensorflow:Assets written to: C:/t\Transform\transform_graph\25\.temp_path\tftransform_tmp\81b9e36648364b8aa20708835bb16d11\assets


INFO:tensorflow:Assets written to: C:/t\Transform\transform_graph\25\.temp_path\tftransform_tmp\81b9e36648364b8aa20708835bb16d11\assets


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 25
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 8. Komponen 6: Tuner (Saran 1 ⭐)
Komponen `Tuner` menjalankan otomatisasi hyperparameter tuning menggunakan modul `tuner.py` dan KerasTuner.

In [8]:
tuner = Tuner(
    module_file='tuner.py',
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=20),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=10)
)
context.run(tuner)


Trial 3 Complete [00h 00m 03s]
val_auc: 0.9402515888214111

Best val_auc So Far: 0.9402515888214111
Total elapsed time: 00h 00m 10s
Results summary
Results in C:/t\.temp\26\heart_disease_tuning
Showing 10 best trials
Objective(name="val_auc", direction="max")

Trial 2 summary
Hyperparameters:
embed_dim_sex: 4
embed_dim_cp: 16
embed_dim_fbs: 4
embed_dim_restecg: 12
embed_dim_exang: 8
embed_dim_slope: 12
embed_dim_ca: 16
embed_dim_thal: 16
units_1: 96
units_2: 16
dropout_rate: 0.30000000000000004
learning_rate: 0.001
Score: 0.9402515888214111

Trial 1 summary
Hyperparameters:
embed_dim_sex: 4
embed_dim_cp: 4
embed_dim_fbs: 16
embed_dim_restecg: 12
embed_dim_exang: 16
embed_dim_slope: 4
embed_dim_ca: 16
embed_dim_thal: 16
units_1: 128
units_2: 64
dropout_rate: 0.1
learning_rate: 0.0005
Score: 0.9025157690048218

Trial 0 summary
Hyperparameters:
embed_dim_sex: 12
embed_dim_cp: 12
embed_dim_fbs: 8
embed_dim_restecg: 12
embed_dim_exang: 4
embed_dim_slope: 12
embed_dim_ca: 12
embed_dim_thal: 

ExecutionResult(
    component_id: Tuner
    execution_id: 26
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 9. Komponen 7: Trainer
`Trainer` melatih model Deep Neural Network (DNN) menggunakan modul `trainer.py` dan hyperparameter terbaik dari `Tuner`.

In [9]:
trainer = Trainer(
    module_file='trainer.py',
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=30),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=10)
)
context.run(trainer)


Instructions for updating:
Use Keras preprocessing layers instead, either directly or via the `tf.keras.utils.FeatureSpace` utility. Each of `tf.feature_column.*` has a functional equivalent in `tf.keras.layers` for feature preprocessing when training a Keras model.


Instructions for updating:
Use Keras preprocessing layers instead, either directly or via the `tf.keras.utils.FeatureSpace` utility. Each of `tf.feature_column.*` has a functional equivalent in `tf.keras.layers` for feature preprocessing when training a Keras model.


Epoch 1/10


 1/30 [>.............................] - ETA: 1:09 - loss: 0.9122 - accuracy: 0.5938 - auc: 0.0000e+00 - precision: 1.0000 - recall: 0.5938

 9/30 [========>.....................] - ETA: 0s - loss: 0.8493 - accuracy: 0.5868 - auc: 0.4747 - precision: 0.9941 - recall: 0.5874      

...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: C:/t\Trainer\model\27\Format-Serving\assets


INFO:tensorflow:Assets written to: C:/t\Trainer\model\27\Format-Serving\assets


ExecutionResult(
    component_id: Trainer
    execution_id: 27
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 10. Komponen 8: Resolver
`Resolver` mengambil model *blessed* sebelumnya dari metadata untuk dijadikan baseline dalam evaluasi model candidate.

In [10]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
)
context.run(model_resolver)


ExecutionResult(
    component_id: Resolver
    execution_id: 28
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Resolver, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 11. Komponen 9: Evaluator
`Evaluator` memvalidasi performa model candidate menggunakan TFMA (`tensorflow_model_analysis`) terhadap ambang batas metrik (Accuracy > 0.75) dan baseline model.

In [11]:
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(label_key='target')
    ],
    slicing_specs=[
        tfma.SlicingSpec(),
        tfma.SlicingSpec(feature_keys=['sex'])
    ],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name='BinaryAccuracy',
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={'value': 0.70}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={'value': -1e-10}
                        )
                    )
                ),
                tfma.MetricConfig(class_name='AUC'),
                tfma.MetricConfig(class_name='Precision'),
                tfma.MetricConfig(class_name='Recall')
            ]
        )
    ]
)

evaluator = Evaluator(
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config,
    examples=example_gen.outputs['examples']
)
context.run(evaluator)


C:\Latihan\AI\Dicoding\MachineLearningPipeline\venv\lib\site-packages\tensorflow_model_analysis\metrics\confusion_matrix_metrics.py:541: RuntimeWarning: invalid value encountered in divide
  fp_rate = fp / (fp + tn)


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 29
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 12. Komponen 10: Pusher
Jika model candidate lolos evaluasi (*Blessed*), `Pusher` mengekspor model ke direktori serving (`serving_model/`).

In [12]:
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    )
)
context.run(pusher)

# Menyinkronkan seluruh artifact komponen pipeline ke direktori sonnyariady-pipeline sesuai kriteria submission
import shutil
if os.name == 'nt' and os.path.exists("C:/t"):
    if os.path.exists(PIPELINE_ROOT):
        shutil.rmtree(PIPELINE_ROOT, ignore_errors=True)
    shutil.copytree("C:/t", PIPELINE_ROOT)

print(f"Pipeline execution complete! Artifacts saved to {PIPELINE_ROOT} and serving model pushed to: {SERVING_MODEL_DIR}")


Pipeline execution complete! Artifacts saved to sonnyariady-pipeline and serving model pushed to: serving_model\heart-disease-model
